<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/mesoscope_sequence_area.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mesoscope sequence-mismatch by cortical area — and how it compares to Neuropixels

The Neuropixels sequence result (Result 4) is a robust positive prediction-error signal
(pooled DvI ≈ +0.21). This notebook breaks the **mesoscope** sequence DvI down by cortical
area (VISp vs VISl) across 16 sessions. The mesoscope pooled number looks weak (+0.04), but
that averages over areas — VISl and VISp differ sharply.

**The important comparison, though, is against Neuropixels broken down the same way** — and
there the two modalities *disagree in V1*:

| area | Neuropixels DvI (90°) | Mesoscope DvI (90°) |
|---|---|---|
| **VISp** (primary V1) | **+0.25** (strongest area) | **−0.08** (negative) |
| **VISl** (lateral) | +0.16 (n.s., n=99) | +0.18 |

Spiking finds the sequence-PE *strongest in V1*; 2-photon finds it *absent/negative in V1*,
positive only laterally. This is a **cross-modality areal discrepancy**, only partly explained
by mesoscope's superficial laminar sampling (it misses L6, where the Neuropixels V1 signal is
largest — but even superficial-matched, spiking V1 stays positive at +0.19 vs 2p −0.08). See
the README "Result 8" section for the full comparison and the laminar analysis. This notebook
computes the mesoscope side; somatic ROIs only (`is_soma`), DvI vs the equiprobable Control
block 2, response window 0–1 s.

In [ ]:
import sys, subprocess
try:
    import remfile, h5py  # noqa
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","remfile","h5py","requests","pandas","numpy","matplotlib","scipy"],check=True)
import numpy as np, pandas as pd, re, h5py, remfile, requests
from scipy import stats as ss
QUICK = True   # True: 4 sessions for a fast Colab pass; False: all 16

def s3(aid, ds="001768"):
    return requests.get(f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/draft/assets/{aid}/download/",allow_redirects=False,timeout=60).headers["Location"]
def resolve_asset(subject, ses_substr, ds="001768"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/draft/assets/"
    r=requests.get(u,params={"path":f"sub-{subject}/"},timeout=30).json()
    hits=[a for a in r["results"] if ses_substr in a["path"]]
    if len(hits)!=1: raise LookupError(f"{subject}/{ses_substr}: {len(hits)} matches")
    return hits[0]["asset_id"]
def dec(a): return np.array([x.decode() if isinstance(x,bytes) else x for x in a])

In [ ]:
SESSIONS = [
 ("843001","2026-04-02"),
 ("832700","2026-02-07"),
 ("843000","2026-03-04"),
 ("839909","2026-03-06"),
 ("839909","2026-03-19"),
 ("843000","2026-03-03"),
 ("837568","2026-02-16"),
 ("832700","2026-02-06"),
 ("837568","2026-02-13"),
 ("843001","2026-04-01"),
 ("845342","2026-04-01"),
 ("846289","2026-04-21"),
 ("842971","2026-04-22"),
 ("846289","2026-04-20"),
 ("845342","2026-03-31"),
 ("842971","2026-04-18"),
]
if QUICK: SESSIONS = SESSIONS[:4]  # result-blind: first 4 in registry order (fast Colab pass)
def extract(aid, subj):
    fh=h5py.File(remfile.File(s3(aid)),"r")
    g=fh["intervals"]["Sequence mismatch block_presentations"]; TT=dec(g["TrialType"][:]); ts=g["start_time"][:]
    tis=dec(g["TrialInSequence"][:]).astype(float)
    gc=fh["intervals"]["Control block 2_presentations"]; cori=dec(gc["Orientation"][:]).astype(float); cts=gc["start_time"][:]
    ctf=dec(gc["TemporalFrequency"][:]).astype(float) if "TemporalFrequency" in gc else np.full(len(cts),2.0)
    dev=ts[(TT=="orientation_90")&(tis==3)]; ctrl=cts[(np.abs(np.degrees(cori)-90)<5)&(np.abs(ctf-2)<0.5)]
    op=fh["general"]["optophysiology"]; rows=[]
    for pl in [k for k in fh["processing"].keys() if k.startswith("VIS")]:
        loc=op[pl]["location"][()]; loc=loc.decode() if isinstance(loc,bytes) else loc
        m=re.search(r"Structure:\s*(\w+)\s+Depth:\s*(\d+)",loc); area=m.group(1); depth=int(m.group(2))
        pr=fh["processing"][pl]; D=pr["dff_timeseries"]["dff_timeseries"]["data"][:]; tt=pr["dff_timeseries"]["dff_timeseries"]["timestamps"][:]
        rt=pr["image_segmentation"]["roi_table"]; is_soma=rt["is_soma"][:].astype(bool) if "is_soma" in rt else np.ones(D.shape[1],bool)
        def wr(on,rw=(0.0,1.0),bw=(-0.5,-0.05)):
            R=np.full((len(on),D.shape[1]),np.nan)
            for i,o in enumerate(on):
                l0=np.searchsorted(tt,o+rw[0]);h0=np.searchsorted(tt,o+rw[1]);lb=np.searchsorted(tt,o+bw[0]);hb=np.searchsorted(tt,o+bw[1])
                if h0>l0 and hb>lb: R[i]=np.nanmean(D[l0:h0],0)-np.nanmean(D[lb:hb],0)
            return np.nanmean(R,0)
        rd=wr(dev); rc=wr(ctrl); mk=is_soma&~(np.isnan(rd)|np.isnan(rc))
        dvi=(rd[mk]-rc[mk])/(np.abs(rd[mk])+np.abs(rc[mk])+1e-9)
        for v in dvi: rows.append((subj,area,depth,v))
    fh.close(); return rows

In [ ]:
rows=[]
for si,(subj,ses) in enumerate(SESSIONS):
    r=extract(resolve_asset(subj,ses),subj)
    rows+=[(*x,si) for x in r]
    print(f"  {si+1}/{len(SESSIONS)} {subj}: {len(r)} somatic ROIs")
DM=pd.DataFrame(rows,columns=["subject","area","depth","dvi","session"])
sm=DM.groupby(["session","area"]).dvi.median().unstack()
for area in ["VISl","VISp"]:
    print(f"{area}: DvI={DM[DM.area==area].dvi.median():+.3f}, {(sm[area]>0).sum()}/{len(sm)} sessions positive")
paired=sm.dropna(); w=ss.wilcoxon(paired["VISl"],paired["VISp"])
print(f"VISl>VISp in {(paired['VISl']>paired['VISp']).sum()}/{len(paired)} sessions, Wilcoxon p={w.pvalue:.4f}")

## Neuropixels side — the same sequence contrast, by area and layer

The mesoscope result above is only half of Result 8. The comparison that carries the finding is
against the **Neuropixels** sequence DvI *by area* (and the depth-matched "superficial" subset).
This cell recomputes it from the ecephys sequence sessions with the **exact device==group probe
mapping** (the substring bug is fixed) and the responsiveness gate used in
`sequence_mismatch_ecephys.ipynb`, then writes `seq_area_comparison.csv` and `seq_visp_layers.csv`.

In [ ]:
import numpy as np, pandas as pd, h5py, remfile, requests, re, time
def s3u(ds,aid): return requests.get(f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/draft/assets/{aid}/download/",allow_redirects=False,timeout=60).headers["Location"]
def colf(g,c): v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])
def resolve_ece(subj,date):
    path=f"sub-{subj}/sub-{subj}_ses-ecephys-{subj}-{date}_ecephys.nwb"
    r=requests.get("https://api.dandiarchive.org/api/dandisets/001637/versions/draft/assets/",params={"path":path},timeout=30)
    return [a for a in r.json()["results"] if a["path"]==path][0]["asset_id"]
ECE_SEQ=[("830794","2026-01-28-11-01-44"),("830846","2026-03-09-10-32-54"),("830849","2026-03-04-10-14-36"),
         ("830848","2026-03-02-15-05-26"),("830852","2026-02-25-10-32-11"),("830851","2026-03-18-10-24-59"),
         ("830795","2026-02-25-16-03-31")]
if QUICK: ECE_SEQ=ECE_SEQ[:3]   # result-blind: first 3 in registry order
RESP=(0.03,0.28); BASE=(-0.1,-0.005)
def seq_np(subj,date):
    fh=h5py.File(remfile.File(s3u("001637",resolve_ece(subj,date))),"r")
    g=fh["intervals"]["Sequence mismatch block_presentations"]; TT=colf(g,"TrialType"); ts=g["start_time"][:]
    cb=None
    for k in fh["intervals"].keys():
        gk=fh["intervals"][k]
        if "BlockType" in gk and "sequential_control_block" in set(colf(gk,"BlockType")): cb=gk; break
    cori=colf(cb,"Orientation").astype(float); cts=cb["start_time"][:]
    U=fh["units"]; ut=U["spike_times"]; uidx=U["spike_times_index"][:]
    qc=U["default_qc"][:] if "default_qc" in U else np.ones(len(uidx),bool)  # QC gate (matches sequence_units.parquet)
    spikes=[ut[0:uidx[0]]]+[ut[uidx[i-1]:uidx[i]] for i in range(1,len(uidx))]
    def rate(sp,ev):
        if len(ev)<3: return np.nan
        return np.nanmean([(np.searchsorted(sp,t+RESP[1])-np.searchsorted(sp,t+RESP[0]))/(RESP[1]-RESP[0])
                           -(np.searchsorted(sp,t+BASE[1])-np.searchsorted(sp,t+BASE[0]))/(BASE[1]-BASE[0]) for t in ev])
    # exact device==group probe mapping
    egrp=colf(fh["general/extracellular_ephys/electrodes"],"group_name"); eloc=colf(fh["general/extracellular_ephys/electrodes"],"location")
    dev=colf(fh["units"],"device_name"); eci=fh["units"]["extremum_channel_index"][:]
    offs={p:int(np.where(egrp==p)[0][0]) for p in set(egrp)}; blk={p:int((egrp==p).sum()) for p in offs}
    uarea=np.array([eloc[offs[dev[i]]+min(int(eci[i]),blk[dev[i]]-1)] for i in range(len(dev))])
    arf=lambda a:(re.match(r"([A-Za-z]+)",str(a)) or [None,str(a)])[1]
    layf=lambda a:(re.match(r"VIS[a-z]*(\d[a-b]?)",str(a)) or [None,""])[1]
    odd90=ts[TT=="orientation_90"]; c90=cts[np.abs(np.degrees(cori)-90)<5]
    rows=[]
    for uid in range(len(spikes)):
        if not qc[uid]: continue                      # keep QC-passing units (as in sequence_units.parquet)
        ro=rate(spikes[uid],odd90); rc=rate(spikes[uid],c90)
        if np.isnan(ro) or np.isnan(rc): continue     # needs >=3 trials each
        rows.append((subj,arf(uarea[uid]),layf(uarea[uid]),(ro-rc)/(abs(ro)+abs(rc)+1e-9)))
    fh.close(); return rows
def bh(v,s,n=5000,seed=42):
    v=np.asarray(v); s=np.asarray(s); ok=~np.isnan(v); v,s=v[ok],s[ok]
    rng=np.random.default_rng(seed); us=np.unique(s); by={x:v[s==x] for x in us}
    m=[np.median(np.concatenate([rng.choice(by[x],len(by[x]),True) for x in rng.choice(us,len(us),True)])) for _ in range(n)]
    return float(np.median(v)),float(np.percentile(m,2.5)),float(np.percentile(m,97.5))
NP=pd.DataFrame([r for subj,date in ECE_SEQ for r in seq_np(subj,date)],columns=["subject","area","layer","dvi90"])
print(f"Neuropixels: {len(NP)} responsive sequence units")
for a in ["VISp","VISl"]:
    m,lo,hi=bh(NP[NP.area==a].dvi90.values,NP[NP.area==a].subject.values)
    print(f"  NP {a}: {m:+.3f} [{lo:+.3f},{hi:+.3f}] n={(NP.area==a).sum()}")
# superficial-matched VISp (L2/3+L4, the depth mesoscope sees)
def lb(L):
    L=str(L); return "L2/3" if L[:1] in ("1","2") else {"4":"L4","5":"L5","6a":"L6a","6b":"L6b"}.get(L,L)
vp=NP[NP.area=="VISp"].copy(); vp["lb"]=vp.layer.map(lb)
sup=vp[vp.lb.isin(["L2/3","L4"])]; ms,los,his=bh(sup.dvi90.values,sup.subject.values)
print(f"  NP VISp superficial (L2/3+L4): {ms:+.3f} [{los:+.3f},{his:+.3f}] n={len(sup)}")
LAM=pd.DataFrame([{"layer":L,"med":vp[vp.lb==L].dvi90.median(),"n":int((vp.lb==L).sum())} for L in ["L2/3","L4","L5","L6a","L6b"]])
# Combined NP-vs-mesoscope area comparison (the Result 8 table). DM (mesoscope per-ROI) is
# from the cell above. Both CSVs reflect the FULL session sets only — a QUICK subset is a
# fast preview, not the authoritative table, so we guard the writes.
comp=[]
for a in ["VISp","VISl"]:
    em,elo,ehi=bh(NP[NP.area==a].dvi90.values,NP[NP.area==a].subject.values)
    dm=DM[DM.area==a]; mm,mlo,mhi=bh(dm.dvi.values,dm.subject.values)
    comp.append(dict(area=a,ece_med=em,ece_lo=elo,ece_hi=ehi,ece_n=int((NP.area==a).sum()),
                     meso_med=mm,meso_lo=mlo,meso_hi=mhi,meso_n=len(dm),
                     ece_sig=bool(elo>0 or ehi<0),meso_sig=bool(mlo>0 or mhi<0)))
COMP=pd.DataFrame(comp)
if not QUICK:
    LAM.to_csv("seq_visp_layers.csv",index=False)      # only the full 7-session set is authoritative
    COMP.to_csv("seq_area_comparison.csv",index=False)
print(LAM.to_string(index=False)); print(); print(COMP.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
acol={"VISl":"#c0392b","VISp":"#1b4079"}
fig,(a1,a2)=plt.subplots(1,2,figsize=(9,4)); fig.subplots_adjust(wspace=0.4,top=0.85,bottom=0.15,left=0.1,right=0.97)
for y,ar in [(1,"VISl"),(0,"VISp")]:
    d=DM[DM.area==ar].dvi; a1.plot(d.median(),y,"o",color=acol[ar],ms=11)
    a1.text(d.median(),y+0.2,f"{ar} {d.median():+.2f}",ha="center",fontweight="bold",color=acol[ar],fontsize=10)
a1.axvline(0,color="k",ls=":",lw=0.8); a1.set_ylim(-0.5,1.6); a1.set_yticks([]); a1.set_xlabel("sequence DvI (90 deg)")
a1.set_title("Sequence-PE in VISl, not VISp",fontsize=9)
for _,rr in sm.iterrows(): a2.plot([0,1],[rr["VISp"],rr["VISl"]],"-",color="0.7",lw=0.8)
a2.scatter([0]*len(sm),sm["VISp"],color=acol["VISp"],s=30); a2.scatter([1]*len(sm),sm["VISl"],color=acol["VISl"],s=30)
a2.axhline(0,color="k",ls=":",lw=0.8); a2.set_xticks([0,1]); a2.set_xticklabels(["VISp","VISl"]); a2.set_ylabel("per-session median DvI")
a2.set_title("Paired VISl > VISp",fontsize=9)
for sp in ["top","right"]:
    a1.spines[sp].set_visible(False); a2.spines[sp].set_visible(False)
plt.show()

### Takeaway

The mesoscope sequence signal is area-specific (VISl positive, VISp negative), but the
scientifically important point is the **comparison to Neuropixels**: spiking finds the
sequence prediction-error *strongest in V1*, whereas 2-photon finds it *absent/negative in V1*.
The apparent VISl agreement (+0.18 vs Neuropixels +0.16) is not something to lean on — the
Neuropixels VISl estimate is not significant (n=99). The honest conclusion is that the calcium
DvI is **not interchangeable** with the spiking DvI, especially in V1. Only part of the gap is
laminar sampling (mesoscope misses deep layers where the V1 signal is largest; but superficial-
matched, spiking V1 is still +0.19 vs 2p −0.08). Candidate causes — indicator nonlinearity/
thresholding, neuropil contamination, or a genuine spiking-vs-calcium transformation — cannot
be separated with this dataset. A per-subject magnitude spread sits on top of the area pattern,
a candidate for a behavioral-state analysis as more sessions accrue.